## Lab 4: API Keys

### Part 1: Using an API Key to Access a Web Service

Create a PDF of a webpage (https://andrewbeatty1.pythonanywhere.com/bookviewer.html).
Get API Key from  html2pdf.com

In [58]:
import requests
import urllib.parse
import json
import base64
from config import apikeys as cfg

API key has been saved separately in a config file. This is a good practice to avoid hardcoding sensitive information in your code.

In [3]:
# Load the API key from the config file
apikey = cfg["htmltopdfkey"]

In [4]:
# Define the target URL and the API endpoint
targeturl = "https://andrewbeatty1.pythonanywhere.com/bookviewer.html"
apiurl = "https://api.html2pdf.app/v1/generate"

In [5]:
# Set up the parameters for the API request
params = {
    "url": targeturl,
    "apiKey": apikey
}
# encode the parameters
parsed_params = urllib.parse.urlencode(params)
# construct the full request URL
request_url = f"{apiurl}?{parsed_params}"

In [7]:
# Make the API request to generate the PDF
response = requests.get(request_url)

In [8]:
print(response.status_code)

200


Save the pdf as a file.

In [9]:
with open ("bookviewer.pdf", "wb") as f:
    f.write(response.content)

### Part 2: Get private data from GitHub using a Personal Access Token (PAT)

In [4]:
# Get API Key from GitHub
githubkey = cfg["aprivateone"]

In [5]:
# Define the GitHub API endpoint and the repository information
apiurl = "https://api.github.com/repos/AnnaLozenko/aprivateone"

In [6]:
# Set up the authentication using the Personal Access Token (PAT)
response = requests.get(apiurl, auth= ("token", githubkey))

In [11]:
# Check the response status code and print the repository information
print(response.status_code)
repoJSON = response.json()

200


If the API key is valid, the response status should be 200. Otherwise, you may get a 401 Unauthorized error if the API key is invalid or missing.

In [10]:
# save the repository information to a JSON file
with open ("repo.json", "w") as f:
    json.dump(repoJSON, f, indent=4)

## Part 3: Script that makes a change to a GitHub repository using the GitHub API and a Personal Access Token (PAT)

Important: To update a file in a GitHub repository using the API, you need to have the necessary permissions to access and modify the repository (read and write). Make sure that your Personal Access Token (PAT) has the appropriate scopes (permissions) to perform the update operation on the repository. If you do not have the required permissions, you may encounter a 403 Forbidden error when trying to update the file.

In [98]:
# Define the GitHub API endpoint for updating a file and the repository information.
# I will attempt to change the README.md file in the repository.
targeturl = "https://api.github.com/repos/AnnaLozenko/aprivateone/contents/README.md"

In [99]:
headers = {
    "Authorization": f"Bearer {githubkey}",
    "Accept": "application/vnd.github+json"
}

In [100]:
# Make the API request to update the README.md file
response = requests.get(targeturl, headers=headers)
print(response.status_code)

200


In [101]:
# To update a file on GitHub using the API, you need to provide the current SHA of the file you want to update. The SHA is a unique identifier for the file's current state. You can get the SHA from the response of the GET request to the file's endpoint.
sha = response.json()["sha"]

In [102]:
# gitHub API requires the content to be base64 encoded when updating a file. So we need to encode the new content before making the API request to update the README.md file.
new_content = "Hello there! This is a new content for the README.md file."
new_content_encoded = base64.b64encode(new_content.encode("utf-8")).decode("utf-8")

In [103]:
# Now we can make the API request to update the README.md file. We need to include the new content, the current SHA of the file, and a commit message in the request payload.
update_payload = {
    "message": "Update README.md file via API",
    "content": new_content_encoded,
    "sha": sha, # Get the current SHA of the file to be updated
    "branch" : "main"
}

In [104]:
# Make the API request to update the README.md file
put_response = requests.put(targeturl, headers=headers, json=update_payload)
print(put_response.status_code)

200
